# Homework 0 - Using k-NN to Analyze Scientific Data

This assignment will be using the elementary KNN classification model to classify data from the penguin dataset. This assignment will focus on performing operations and manipulations on the data to get clear insight.

In [ ]:
# Adding all dependencies to the environment
import pandas as pd  # Data manipulation and analysis
import scipy as sc  # Scientific Computing
import numpy as np  # Numerical Computing
import matplotlib.pyplot as plt  # Data Visualization with Static Plots
import plotly.graph_objects as go  # Interative Data Visualization
import math  # Standard Library for Math Operations
from sklearn.model_selection import train_test_split  # Data Splitting
from sklearn.neighbors import KNeighborsClassifier  # ML Models

## Problem 1 - Loading the dataset

Loading the penguins dataset (*peguins_size*) into a dataframe.

In [47]:
df = pd.read_csv('data/penguins_size.csv')  # Load the csv file into a pandas dataframe

## Problem 2 - Cleaning the data

Firstly, examining the dataframe.

In [ ]:
print(f"Columns:\n{df.dtypes}\n")
print(f"Description:\n{df.describe}")

Columns:
species               object
island                object
culmen_length_mm     float64
culmen_depth_mm      float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
species_type         float64
sex_type             float64
dtype: object

DF Count: <bound method DataFrame.count of     species     island  culmen_length_mm  culmen_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen              39.1             18.7              181.0   
1    Adelie  Torgersen              39.5             17.4              186.0   
2    Adelie  Torgersen              40.3             18.0              195.0   
4    Adelie  Torgersen              36.7             19.3              193.0   
5    Adelie  Torgersen              39.3             20.6              190.0   
..      ...        ...               ...              ...                ...   
338  Gentoo     Biscoe              47.2             13.7              214.0   
340  Gentoo     Biscoe          

Removing any incomplete records.

In [49]:
df = df.dropna(how="any")  # Dropping any incomplete data
df

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


Verifying that the new dataframe only contains complete records.

In [50]:
print(df.isna().sum())  # Obtain a count of rows that contain an incomplete record

df  # Display the full dataframe

species              0
island               0
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


## Problem 3 - Integer classifications for columns

Create integer categorizations for the original species and sex fields, named species_type and sex_type.

First, determine the unique names for species and sex.

In [56]:
# Obtain the unique names for species and sex
species_names = df['species'].unique()
sex_names = df['sex'].unique()
print(f"Species Names: {species_names}, Sex Names: {sex_names}")  # Quick manual verification

Species Names: ['Adelie' 'Chinstrap' 'Gentoo'], Sex Names: ['MALE' 'FEMALE' '.']


Next, categorize these names into integer representations, add to the df, and convert from floats to ints.

In [57]:

# Create new df columns for species types and sex types
for i in range(len(species_names)):
    df.loc[df['species'] == species_names[i], 'species_type'] = i
for j in range(len(sex_names)):
    df.loc[df['sex'] == sex_names[j], 'sex_type'] = j
    
# Converting the float columns to ints
df = df.astype({'species_type': 'int32', 'sex_type': 'int32'})
    
# Verify the new df with a random sample
df.sample(10)

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex,species_type,sex_type
178,Chinstrap,Dream,50.5,18.4,200.0,3400.0,FEMALE,1,1
337,Gentoo,Biscoe,48.8,16.2,222.0,6000.0,MALE,2,0
137,Adelie,Dream,40.2,20.1,200.0,3975.0,MALE,0,0
271,Gentoo,Biscoe,48.5,14.1,220.0,5300.0,MALE,2,0
107,Adelie,Biscoe,38.2,20.0,190.0,3900.0,MALE,0,0
319,Gentoo,Biscoe,51.1,16.5,225.0,5250.0,MALE,2,0
309,Gentoo,Biscoe,52.1,17.0,230.0,5550.0,MALE,2,0
69,Adelie,Torgersen,41.8,19.4,198.0,4450.0,MALE,0,0
64,Adelie,Biscoe,36.4,17.1,184.0,2850.0,FEMALE,0,1
53,Adelie,Biscoe,42.0,19.5,200.0,4050.0,MALE,0,0


## Problem 4 - Feature Selection

First, isolate the features and target.

In [ ]:
y_df = df['species_type']  # Target
x_df = df.drop(columns=['species_type', 'species', 'sex'])  # Features


In [ ]:

# Split into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x_df, x_df, test_size=.25, random_state=42)

## References (Problem 10)

- [Pandas Online Documentation - Missing Data](https://pandas.pydata.org/docs/user_guide/10min.html#missing-data)
- [Pandas Online Documentation - Where Method](https://pandas.pydata.org/docs/user_guide/indexing.html#the-where-method-and-masking)